# 카드 지출 절감 제안기 (Group 03)

**알고리즘과 컴퓨터적 사고 · Project 2** — 고준호(팀장) · 정하나 · 안태현

## 문제 정의
카드값이 많이 나오는데 어디서 새는지 모른다. 매일 쓰는 가계부는 이미 포기했고,
월 1회 몰아서 확인하고 싶다. 그래서 **최근 3개월 카드 내역을 비교해,
이번 달 줄일 재량 지출 1개와 절감 여지 금액을 과거 실적 근거와 함께** 제시한다.

## 사용자 · 목표 · 제약
| 항목 | 내용 |
|---|---|
| 사용자 | 카드값이 많이 나오지만 어디서 새는지 모르는 20대 대학생 1명 |
| 목표 | "이번 달엔 이걸 줄이면 된다"를 **한 문장**으로 받기 |
| 제약 | 외부 라이브러리·머신러닝·실제 CSV 읽기·실시간 연동 없음. List · Dict · if · for · 함수만 사용 |
| 제약 | 입력은 **더미 데이터**(가상 거래 내역). 개인 카드 정보를 쓰지 않는다 |

## 성공 기준
1. 3개월치 더미 거래를 넣고 "모두 실행"하면 제안 문장이 **정확히 1줄** 나온다.
2. 제안에는 카테고리 · 이번 달 지출 · 3개월 최저 지출(해당 월) · 절감 여지 · 반복 결제 비율이 모두 담긴다.
3. 절감 여지가 없는 달, 거래가 0건인 달, 값이 누락된 거래가 섞인 경우에도 **에러 없이** 문장 1줄을 낸다.

## 파이프라인 (IPO)
```
transactions ──C1──▶ cleaned ──C2(+label_rules)──▶ labeled ──C3──▶ labeled(보정)
                                                                     │
                     ┌───────────────────────────────────────────────┤
                     ▼                                               ▼
               C4 → monthly_totals ──▶ C5(+labeled, this_month) ──▶ savings ──C6──▶ 제안 문장 1줄
```


## 0. 더미 데이터

실제 카드 CSV를 읽지 않는다. 아래 `transactions`와 `label_rules`는 **손으로 만든
더미 데이터(가상 거래 내역)** 이며, 2026-07 · 2026-08 · 2026-09 세 달을 담는다.
이번 달은 2026-09다. 담당: 고준호


In [ ]:
"""더미 데이터 — 실제 카드 CSV를 읽지 않고 손으로 만든 가상 거래 내역.

Input  : 없음 (모듈 상수)
Process: 2026-07 ~ 2026-09 3개월치 더미 거래와 라벨 규칙표를 정의한다
Output : transactions, label_rules, transactions_no_saving, transactions_missing

주의: 아래 금액은 plan.md 9장의 목표 수치(C1 정제 후 기준)에 맞춰 역산한 값이다.
      금액이나 날짜를 고치면 plan.md 10장의 기대 문장과 테스트가 깨진다.

C1 정제 후 재량 카테고리 월 합계
  카페·간식  2026-07: 120,000 / 2026-08: 150,000 / 2026-09: 187,000  → 절감 여지 67,000 (1위)
  식비       2026-07: 200,000 / 2026-08: 180,000 / 2026-09: 210,000  → 절감 여지 30,000 (2위)
  쇼핑       2026-07:  95,000 / 2026-08:  78,000 / 2026-09:  42,000  → 이번 달 최저 → 제외
"""

# ── 라벨 규칙표 (더미) ────────────────────────────────────────────────
# 위에서부터 순회해 keyword in merchant인 첫 규칙을 적용하므로 순서가 우선순위다.
# 예: "쿠팡이츠"(식비)를 "쿠팡"(쇼핑)보다 반드시 위에 둔다.
label_rules = [
    # 식비 (재량)
    {"keyword": "쿠팡이츠", "category": "식비", "is_discretionary": True},
    {"keyword": "배달의민족", "category": "식비", "is_discretionary": True},
    {"keyword": "학생식당", "category": "식비", "is_discretionary": True},
    {"keyword": "김밥", "category": "식비", "is_discretionary": True},
    {"keyword": "곱창", "category": "식비", "is_discretionary": True},
    {"keyword": "갈비", "category": "식비", "is_discretionary": True},
    {"keyword": "치킨", "category": "식비", "is_discretionary": True},
    {"keyword": "마라탕", "category": "식비", "is_discretionary": True},
    # 카페·간식 (재량)
    {"keyword": "스타벅스", "category": "카페·간식", "is_discretionary": True},
    {"keyword": "메가커피", "category": "카페·간식", "is_discretionary": True},
    {"keyword": "이디야", "category": "카페·간식", "is_discretionary": True},
    {"keyword": "투썸", "category": "카페·간식", "is_discretionary": True},
    {"keyword": "공차", "category": "카페·간식", "is_discretionary": True},
    {"keyword": "배스킨라빈스", "category": "카페·간식", "is_discretionary": True},
    {"keyword": "파리바게뜨", "category": "카페·간식", "is_discretionary": True},
    # 쇼핑 (재량) — "쿠팡"은 반드시 "쿠팡이츠"보다 아래에 있어야 한다
    {"keyword": "쿠팡", "category": "쇼핑", "is_discretionary": True},
    {"keyword": "무신사", "category": "쇼핑", "is_discretionary": True},
    {"keyword": "올리브영", "category": "쇼핑", "is_discretionary": True},
    # 교통 (고정)
    {"keyword": "티머니", "category": "교통", "is_discretionary": False},
    {"keyword": "카카오T", "category": "교통", "is_discretionary": False},
    {"keyword": "코레일", "category": "교통", "is_discretionary": False},
    # 구독료 (고정)
    {"keyword": "넷플릭스", "category": "구독료", "is_discretionary": False},
    {"keyword": "유튜브", "category": "구독료", "is_discretionary": False},
    {"keyword": "멜론", "category": "구독료", "is_discretionary": False},
]

# ── 기본 더미 거래 내역 ───────────────────────────────────────────────
transactions = [
    # ===== 2026-07 =====
    {"date": "2026-07-01", "merchant": "학생식당", "amount": 5000},
    {"date": "2026-07-03", "merchant": "티머니 충전", "amount": 50000},
    {"date": "2026-07-04", "merchant": "쿠팡이츠", "amount": 38000},
    {"date": "2026-07-06", "merchant": "메가커피 왕십리점", "amount": 4500},
    {"date": "2026-07-10", "merchant": "넷플릭스", "amount": 13500},
    {"date": "2026-07-11", "merchant": "스타벅스 한양대점", "amount": 11200},
    {"date": "2026-07-13", "merchant": "배달의민족", "amount": 42000},
    {"date": "2026-07-14", "merchant": "동네문방구", "amount": 3200},           # 미매칭 → 기타(고정)
    {"date": "2026-07-15", "merchant": "쿠팡 와우멤버십", "amount": 7890},      # C3 재분류 대상
    {"date": "2026-07-16", "merchant": "공차 한양대점", "amount": 18300},
    {"date": "2026-07-19", "merchant": "투썸플레이스 왕십리점", "amount": 32000},
    {"date": "2026-07-20", "merchant": "무신사", "amount": 68000},
    {"date": "2026-07-22", "merchant": "카카오T", "amount": 8900},
    {"date": "2026-07-23", "merchant": "파리바게뜨 한양대점", "amount": 24000},
    {"date": "2026-07-24", "merchant": "한양곱창", "amount": 65000},
    {"date": "2026-07-25", "merchant": "올리브영 왕십리점", "amount": 27000},
    {"date": "2026-07-27", "merchant": "명륜진사갈비 왕십리점", "amount": 50000},
    {"date": "2026-07-30", "merchant": "배스킨라빈스 왕십리점", "amount": 30000},

    # ===== 2026-08 =====
    {"date": "2026-08-01", "merchant": "학생식당", "amount": 5000},
    {"date": "2026-08-04", "merchant": "티머니 충전", "amount": 50000},
    {"date": "2026-08-05", "merchant": "쿠팡이츠", "amount": 35000},
    {"date": "2026-08-06", "merchant": "메가커피 왕십리점", "amount": 4500},
    {"date": "2026-08-09", "merchant": "스타벅스 한양대점", "amount": 16800},
    {"date": "2026-08-10", "merchant": "넷플릭스", "amount": 13500},
    {"date": "2026-08-13", "merchant": "배달의민족", "amount": 40000},
    {"date": "2026-08-14", "merchant": "이디야커피 사근동점", "amount": 7000},
    {"date": "2026-08-15", "merchant": "쿠팡 와우멤버십", "amount": 7890},      # C3 재분류 대상
    {"date": "2026-08-18", "merchant": "공차 한양대점", "amount": 18700},
    {"date": "2026-08-19", "merchant": "무신사", "amount": 57000},
    {"date": "2026-08-21", "merchant": "파리바게뜨 한양대점", "amount": 29000},
    {"date": "2026-08-22", "merchant": "카카오T", "amount": 12300},
    {"date": "2026-08-24", "merchant": "투썸플레이스 왕십리점", "amount": 38000},
    {"date": "2026-08-26", "merchant": "한양곱창", "amount": 55000},
    {"date": "2026-08-28", "merchant": "올리브영 왕십리점", "amount": 21000},
    {"date": "2026-08-30", "merchant": "배스킨라빈스 왕십리점", "amount": 36000},
    {"date": "2026-08-31", "merchant": "명륜진사갈비 왕십리점", "amount": 45000},

    # ===== 2026-09 (이번 달) =====
    {"date": "2026-09-01", "merchant": "학생식당", "amount": 5000},
    {"date": "2026-09-02", "merchant": "투썸플레이스 왕십리점", "amount": 38000},
    {"date": "2026-09-03", "merchant": "티머니 충전", "amount": 50000},
    {"date": "2026-09-04", "merchant": "쿠팡이츠", "amount": 32000},
    {"date": "2026-09-05", "merchant": "스타벅스 한양대점", "amount": 5600},    # 기획서 샘플
    {"date": "2026-09-05", "merchant": "스타벅스 한양대점", "amount": 5600},    # 완전 중복 → C1 제거
    {"date": "2026-09-06", "merchant": "메가커피 왕십리점", "amount": 4500},
    {"date": "2026-09-08", "merchant": "메가커피 왕십리점", "amount": 4500},
    {"date": "2026-09-09", "merchant": "스타벅스 한양대점", "amount": 5600},
    {"date": "2026-09-10", "merchant": "넷플릭스", "amount": 13500},            # 기획서 샘플
    {"date": "2026-09-11", "merchant": "배달의민족", "amount": 40000},
    {"date": "2026-09-12", "merchant": "파리바게뜨 한양대점", "amount": 33000},
    {"date": "2026-09-13", "merchant": "무신사", "amount": 25000},
    {"date": "2026-09-14", "merchant": "학생식당", "amount": 5000},
    {"date": "2026-09-15", "merchant": "메가커피 왕십리점", "amount": 9000},
    {"date": "2026-09-15", "merchant": "쿠팡 와우멤버십", "amount": 7890},      # C3 재분류 대상
    {"date": "2026-09-16", "merchant": "스타벅스 한양대점", "amount": 11200},
    {"date": "2026-09-17", "merchant": "무신사", "amount": -25000},             # 취소 건 → C1 제거
    {"date": "2026-09-18", "merchant": "배스킨라빈스 왕십리점", "amount": 19800},
    {"date": "2026-09-19", "merchant": "카카오T", "amount": 9800},
    {"date": "2026-09-20", "merchant": "올리브영 왕십리점", "amount": 17000},
    {"date": "2026-09-21", "merchant": "학생식당", "amount": 5000},
    {"date": "2026-09-22", "merchant": "메가커피 왕십리점", "amount": 13500},
    {"date": "2026-09-23", "merchant": "스타벅스 한양대점", "amount": 16800},
    {"date": "2026-09-24", "merchant": "한양곱창", "amount": 55000},
    {"date": "2026-09-25", "merchant": "공차 한양대점", "amount": 18500},
    {"date": "2026-09-26", "merchant": "학교 복사실", "amount": 1500},          # 미매칭 → 기타(고정)
    {"date": "2026-09-27", "merchant": "명륜진사갈비 왕십리점", "amount": 40000},
    {"date": "2026-09-28", "merchant": "이디야커피 사근동점", "amount": 7000},
    {"date": "2026-09-30", "merchant": "쿠팡이츠", "amount": 28000},
]

# ── 테스트용 더미 데이터 1: 후보 0개 ──────────────────────────────────
# 모든 재량 카테고리에서 2026-09가 3개월 중 최저 → savings가 빈 리스트가 된다.
transactions_no_saving = [
    {"date": "2026-07-05", "merchant": "스타벅스 한양대점", "amount": 50000},
    {"date": "2026-07-10", "merchant": "넷플릭스", "amount": 13500},
    {"date": "2026-07-12", "merchant": "쿠팡이츠", "amount": 60000},
    {"date": "2026-07-20", "merchant": "무신사", "amount": 30000},
    {"date": "2026-08-05", "merchant": "스타벅스 한양대점", "amount": 40000},
    {"date": "2026-08-10", "merchant": "넷플릭스", "amount": 13500},
    {"date": "2026-08-12", "merchant": "쿠팡이츠", "amount": 55000},
    {"date": "2026-08-21", "merchant": "무신사", "amount": 30000},
    {"date": "2026-09-05", "merchant": "스타벅스 한양대점", "amount": 10000},
    {"date": "2026-09-10", "merchant": "넷플릭스", "amount": 13500},
    {"date": "2026-09-12", "merchant": "쿠팡이츠", "amount": 20000},
    {"date": "2026-09-22", "merchant": "무신사", "amount": 10000},
]

# ── 테스트용 더미 데이터 2: 값 누락·잘못된 입력 ───────────────────────
# C1이 앞 5건을 전부 걸러내고, 정상 3건만 남아 문장 1줄이 나와야 한다.
transactions_missing = [
    {"date": "2026-09-01", "merchant": "스타벅스 한양대점"},                     # amount 키 누락
    {"date": None, "merchant": "메가커피 왕십리점", "amount": 4500},             # date가 None
    {"date": "2026-09-03", "merchant": "", "amount": 5000},                      # merchant 빈 문자열
    {"date": "2026-09-04", "merchant": "쿠팡이츠", "amount": "15000"},           # amount가 문자열
    {"date": "20260905", "merchant": "투썸플레이스 왕십리점", "amount": 6000},    # date 형식 오류
    {"date": "2026-07-10", "merchant": "스타벅스 한양대점", "amount": 3000},      # 정상
    {"date": "2026-08-10", "merchant": "스타벅스 한양대점", "amount": 4000},      # 정상
    {"date": "2026-09-10", "merchant": "스타벅스 한양대점", "amount": 9000},      # 정상
]

## 1. C1 `clean_transactions` — 내역 정제

| 항목 | 내용 |
|---|---|
| Input | transactions (List of Dict) |
| Process | 값 누락 · 날짜 형식 오류 · 금액 0 이하(취소) · 완전 중복 거래를 제외 |
| Output | cleaned (List of Dict) |

담당: 안태현


In [ ]:
"""C1 내역 정제 — 담당: 안태현 / 지원: 고준호 / 리뷰: 정하나."""

REQUIRED_KEYS = ["date", "merchant", "amount"]   # 거래 1건에 반드시 있어야 하는 키


def clean_transactions(transactions):
    """
    Input  : transactions (List of Dict) — 원본 카드 거래 내역
    Process: 값 누락·날짜 형식 오류·금액 0 이하(취소)·완전 중복 거래를 제외한다
    Output : cleaned (List of Dict) — 원래 순서를 유지한 정상 거래 목록

    제외 규칙 (에러를 내지 않고 조용히 건너뛴다)
      1. date/merchant/amount 키가 없거나 값이 None 또는 빈 문자열
      2. amount가 int가 아님 (bool도 제외)
      3. amount가 0 이하 (취소 건)
      4. date가 "YYYY-MM-DD" 형식이 아님
      5. (date, merchant, amount)가 앞에 나온 거래와 완전히 같음 (중복 건)
    """
    cleaned = []
    seen = []   # 이미 통과한 (date, merchant, amount) 조합

    for transaction in transactions:
        # 1. 필수 키 존재 여부와 값 누락 확인
        has_all_keys = True
        for key in REQUIRED_KEYS:
            if key not in transaction:
                has_all_keys = False
            elif transaction[key] is None or transaction[key] == "":
                has_all_keys = False
        if not has_all_keys:
            continue

        date = transaction["date"]
        merchant = transaction["merchant"]
        amount = transaction["amount"]

        # 2. amount 자료형 확인 (True/False는 int로 취급되므로 따로 막는다)
        if isinstance(amount, bool) or not isinstance(amount, int):
            continue

        # 3. 취소 건 제외
        if amount <= 0:
            continue

        # 4. 날짜 형식 확인 — 길이 10, 5번째와 8번째 글자가 '-'
        if not isinstance(date, str):
            continue
        if len(date) != 10 or date[4] != "-" or date[7] != "-":
            continue

        # 5. 완전 중복 제외 — 처음 나온 1건만 남긴다
        key_tuple = (date, merchant, amount)
        if key_tuple in seen:
            continue
        seen.append(key_tuple)

        cleaned.append(dict(transaction))

    return cleaned

## 2. C2 `label_transactions` — 카테고리 라벨링

| 항목 | 내용 |
|---|---|
| Input | cleaned + label_rules |
| Process | 가맹점명에 키워드가 있는 첫 규칙으로 카테고리·재량 여부 부여, 미매칭은 "기타"(고정). 날짜를 월·일로 분리 |
| Output | labeled (List of Dict) |

담당: 고준호


In [ ]:
"""C2 카테고리 라벨링 — 담당: 고준호 / 지원: 정하나 / 리뷰: 안태현."""

FALLBACK_CATEGORY = "기타"   # 규칙표에 걸리지 않은 거래의 카테고리 (고정 지출로 본다)


def label_transactions(cleaned, label_rules):
    """
    Input  : cleaned (List of Dict), label_rules (List of Dict)
    Process: 가맹점명에 키워드가 들어 있는 첫 규칙으로 카테고리·재량 여부를 정하고,
             날짜를 거래월(month)과 일(day)로 분리한다
    Output : labeled (List of Dict) — month, day, merchant, category,
             is_discretionary, amount 필드를 가진 거래 목록
    """
    labeled = []

    for transaction in cleaned:
        merchant = transaction["merchant"]
        date = transaction["date"]

        # 규칙표를 위에서부터 순회해 첫 매칭에서 멈춘다 (규칙표 순서 = 우선순위)
        category = FALLBACK_CATEGORY
        is_discretionary = False
        for rule in label_rules:
            if rule["keyword"] in merchant:
                category = rule["category"]
                is_discretionary = rule["is_discretionary"]
                break

        labeled.append({
            "month": date[:7],            # "2026-09"
            "day": int(date[8:10]),       # 5
            "merchant": merchant,
            "category": category,
            "is_discretionary": is_discretionary,
            "amount": transaction["amount"],
        })

    return labeled

## 3. C3 `mark_fixed_repeats` — 고정 반복 결제 보정

| 항목 | 내용 |
|---|---|
| Input | labeled |
| Process | 3개월 내내 월 1건 · 같은 일 · 같은 금액인 가맹점을 고정 지출로 변경 (category는 유지) |
| Output | 보정된 labeled |

담당: 고준호


In [ ]:
"""C3 고정 반복 결제 보정 — 담당: 고준호 / 지원: 안태현 / 리뷰: 정하나."""

TARGET_MONTH_COUNT = 3   # "최근 3개월" — 대상 월이 3개가 아니면 보정하지 않는다


def mark_fixed_repeats(labeled):
    """
    Input  : labeled (List of Dict) — C2 출력
    Process: 3개월 내내 월 1건·같은 일·같은 금액으로 결제된 가맹점을 고정 지출로 바꾼다
             (구독료처럼 매달 똑같이 빠져나가는 돈은 줄일 여지가 없기 때문)
    Output : 보정된 labeled (List of Dict) — 입력을 수정하지 않은 새 리스트

    category는 그대로 두고 is_discretionary만 False로 바꾼다.
    """
    # 입력을 건드리지 않도록 거래마다 복사본을 만든다
    result = []
    for transaction in labeled:
        result.append(dict(transaction))

    # 1. 대상 월 집합 — 3개가 아니면 아무것도 바꾸지 않는다
    months = []
    for transaction in result:
        if transaction["month"] not in months:
            months.append(transaction["month"])
    if len(months) != TARGET_MONTH_COUNT:
        return result

    # 2. 가맹점별로 (month, day, amount) 목록을 모은다
    by_merchant = {}
    for transaction in result:
        merchant = transaction["merchant"]
        if merchant not in by_merchant:
            by_merchant[merchant] = []
        by_merchant[merchant].append(transaction)

    # 3. 세 조건을 모두 만족하는 가맹점을 찾아 고정으로 바꾼다
    for merchant in by_merchant:
        items = by_merchant[merchant]
        if _is_fixed_repeat(items, months):
            for transaction in items:
                transaction["is_discretionary"] = False

    return result


def _is_fixed_repeat(items, months):
    """한 가맹점의 거래 목록이 고정 반복 결제 조건을 모두 만족하는지 판정한다."""
    # 조건 1: 3개 대상 월에 각각 정확히 1건씩 (월 1건 조건 — 매일 가는 카페의 오분류 방지)
    for month in months:
        count = 0
        for transaction in items:
            if transaction["month"] == month:
                count += 1
        if count != 1:
            return False

    # 조건 2·3: 모든 거래의 day와 amount가 같다
    first_day = items[0]["day"]
    first_amount = items[0]["amount"]
    for transaction in items:
        if transaction["day"] != first_day:
            return False
        if transaction["amount"] != first_amount:
            return False

    return True

## 4. C4 `sum_by_month_category` — 월·카테고리 합계

| 항목 | 내용 |
|---|---|
| Input | labeled |
| Process | 재량 거래만 카테고리 × 거래월로 합산 |
| Output | monthly_totals (Dict of Dict) |

담당: 정하나


In [ ]:
"""C4 월·카테고리 합계 — 담당: 정하나 / 지원: 고준호 / 리뷰: 안태현."""


def sum_by_month_category(labeled):
    """
    Input  : labeled (List of Dict) — C3 보정까지 끝난 거래 목록
    Process: 재량 지출(is_discretionary=True) 거래만 카테고리 × 거래월로 금액을 더한다
    Output : monthly_totals (Dict of Dict) — {카테고리: {거래월: 합계}}

    재량 거래가 하나도 없으면 빈 dict를 반환한다.
    """
    monthly_totals = {}

    for transaction in labeled:
        # 고정 지출(교통·구독료·기타, C3가 고정으로 바꾼 거래)은 줄일 대상이 아니라 건너뛴다
        if not transaction["is_discretionary"]:
            continue

        category = transaction["category"]
        month = transaction["month"]

        if category not in monthly_totals:
            monthly_totals[category] = {}
        if month not in monthly_totals[category]:
            monthly_totals[category][month] = 0

        monthly_totals[category][month] += transaction["amount"]

    return monthly_totals

## 5. `calc_repeat_ratio` — 반복 결제 비율 (C5 보조)

| 항목 | 내용 |
|---|---|
| Input | labeled, category, month |
| Process | 2회 이상 방문한 가맹점의 거래 건수 ÷ 전체 거래 건수 |
| Output | 비율 (float) |

담당: 안태현


In [ ]:
"""반복 결제 비율 (C5 보조) — 담당: 안태현 / 지원: 정하나 / 리뷰: 고준호."""


def calc_repeat_ratio(labeled, category, month):
    """
    Input  : labeled (List of Dict), category (str), month (str, "YYYY-MM")
    Process: 해당 카테고리·거래월의 재량 거래 중, 2회 이상 방문한 가맹점의 건수 비중을 센다
             (같은 곳을 반복해서 가는 습관일수록 줄이기 쉽다고 보는 근거)
    Output : 반복 결제 비율 (float, 0.0 ~ 1.0) — 거래가 0건이면 0.0
    """
    # 1. 조건에 맞는 재량 거래만 추린다
    picked = []
    for transaction in labeled:
        if transaction["category"] != category:
            continue
        if transaction["month"] != month:
            continue
        if not transaction["is_discretionary"]:
            continue
        picked.append(transaction)

    # 2. 0건이면 0으로 나누지 않도록 바로 0.0을 반환한다
    if len(picked) == 0:
        return 0.0

    # 3. 가맹점별 거래 건수를 센다 (가맹점명 문자열 전체 기준)
    counts = {}
    for transaction in picked:
        merchant = transaction["merchant"]
        if merchant not in counts:
            counts[merchant] = 0
        counts[merchant] += 1

    # 4. 2회 이상인 가맹점들의 거래 건수 합 ÷ 전체 건수
    repeat_count = 0
    for merchant in counts:
        if counts[merchant] >= 2:
            repeat_count += counts[merchant]

    return repeat_count / len(picked)

## 6. C5 `calc_savings` — 절감 여지와 점수

| 항목 | 내용 |
|---|---|
| Input | monthly_totals + labeled + this_month |
| Process | 이번 달 − 3개월 최저 = 절감 여지 → 0 이하 제외 → 비율 → 점수(0.6×절감비율 + 0.4×반복비율) |
| Output | savings (List of Dict) |

담당: 정하나


In [ ]:
"""C5 절감 여지·점수 계산 — 담당: 정하나 / 지원: 안태현 / 리뷰: 고준호."""


SAVING_WEIGHT = 0.6   # 절감 여지 비율 가중치
REPEAT_WEIGHT = 0.4   # 반복 결제 비율 가중치


def calc_savings(monthly_totals, labeled, this_month):
    """
    Input  : monthly_totals (Dict of Dict), labeled (List of Dict), this_month (str)
    Process: 카테고리마다 이번 달 지출과 3개월 최저 지출을 비교해 절감 여지를 구하고,
             절감 여지 비율과 반복 결제 비율을 가중 합산해 점수를 매긴다
             (절감 여지가 0 이하인 카테고리는 제외 — 0으로 나누는 것도 함께 막힌다)
    Output : savings (List of Dict) — category, this_month, min_month, min_amount,
             saving, saving_ratio, repeat_ratio, score
    """
    savings = []

    for category in monthly_totals:
        month_totals = monthly_totals[category]

        # 이번 달 지출 (해당 카테고리 지출이 없는 달은 0)
        this_amount = month_totals.get(this_month, 0)

        # 최저 지출 월 — 동액이면 가장 이른 달을 택하도록 월 이름 순으로 본다
        min_month = None
        min_amount = 0
        for month in sorted(month_totals):
            if min_month is None or month_totals[month] < min_amount:
                min_month = month
                min_amount = month_totals[month]

        # 절감 여지가 없으면 제안 후보에서 빼고, 이번 달 지출 0인 경우도 여기서 걸러진다
        saving = this_amount - min_amount
        if saving <= 0:
            continue

        saving_ratio = saving / this_amount
        repeat_ratio = calc_repeat_ratio(labeled, category, this_month)
        score = SAVING_WEIGHT * saving_ratio + REPEAT_WEIGHT * repeat_ratio

        savings.append({
            "category": category,
            "this_month": this_amount,
            "min_month": min_month,
            "min_amount": min_amount,
            "saving": saving,
            "saving_ratio": saving_ratio,
            "repeat_ratio": repeat_ratio,
            "score": score,          # 표시용 반올림은 C6에서만 한다
        })

    return savings

## 7. C6 `make_suggestion` — 제안 문장 생성

| 항목 | 내용 |
|---|---|
| Input | savings |
| Process | 점수 내림차순 1위 1건만 문장으로 조립, 후보가 없으면 "절감 여지 없음" |
| Output | 제안 문장 (str) |

담당: 안태현


In [ ]:
"""C6 제안 문장 생성 — 담당: 안태현 / 지원: 고준호 / 리뷰: 정하나."""

NO_SAVING_MESSAGE = "[제안] 이번 달 절감 여지 없음"


def make_suggestion(savings):
    """
    Input  : savings (List of Dict) — C5 출력
    Process: 점수 내림차순으로 정렬해 1위 1건만 근거가 담긴 한 문장으로 만든다
    Output : 제안 문장 (str) — 후보가 없으면 "[제안] 이번 달 절감 여지 없음"
    """
    # 후보가 하나도 없는 경우 (모든 카테고리에서 이번 달이 최저)
    if len(savings) == 0:
        return NO_SAVING_MESSAGE

    # 점수 내림차순 → 동점이면 절감 여지 큰 순 → 그래도 같으면 카테고리명 순
    ranked = sorted(savings, key=_sort_key)
    best = ranked[0]

    return (
        "[제안] {category}: 이번 달 {this_month:,}원, "
        "최근 3개월 최저 {min_amount:,}원({min_month}) → "
        "절감 여지 {saving:,}원, 반복 결제 {repeat_percent}%"
    ).format(
        category=best["category"],
        this_month=best["this_month"],
        min_amount=best["min_amount"],
        min_month=best["min_month"],
        saving=best["saving"],
        repeat_percent=round(best["repeat_ratio"] * 100),
    )


def _sort_key(item):
    """정렬 키 — 점수 내림차순, 절감 여지 내림차순, 카테고리명 오름차순."""
    return (-item["score"], -item["saving"], item["category"])

## 8. `main` — C1~C6 연결

| 항목 | 내용 |
|---|---|
| Input | 없음 (위 더미 데이터 사용) |
| Process | C1 → C2 → C3 → C4 → C5 → C6 순서로 호출 |
| Output | 제안 문장 1줄 출력 |

담당: 정하나 (통합·디버깅: 고준호)


In [ ]:
"""main — C1~C6 연결 / 담당: 정하나 (통합·디버깅: 고준호)."""


EXPECTED_MONTH_COUNT = 3   # plan.md 8장 결정사항 3 — 더미 데이터는 2026-07~09 세 달


def run_pipeline(transactions, label_rules):
    """
    Input  : transactions (List of Dict), label_rules (List of Dict)
    Process: C1 정제 → C2 라벨링 → C3 고정 보정 → C4 월별 합계 → C5 점수 → C6 문장
    Output : 제안 문장 (str) — print하지 않고 문자열로 돌려준다 (테스트 재사용 목적)
    """
    cleaned = clean_transactions(transactions)
    labeled = label_transactions(cleaned, label_rules)
    labeled = mark_fixed_repeats(labeled)

    # 이번 달 = 데이터에 있는 가장 늦은 거래월
    months = []
    for transaction in labeled:
        if transaction["month"] not in months:
            months.append(transaction["month"])

    # 정상 거래가 하나도 없으면 후보 0개로 보고 C6에 빈 리스트를 넘긴다
    if len(months) == 0:
        return make_suggestion([])

    # 거래월이 3개가 아니면 "최근 3개월" 전제가 깨진 것이다.
    # 이때 C3 고정 보정이 멈추고 this_month도 엉뚱한 달이 되어 결과가 조용히 틀리므로,
    # 제안 문장 대신 원인을 알려 준다 (더미 데이터의 date 오타를 바로 찾기 위함).
    if len(months) != EXPECTED_MONTH_COUNT:
        return (
            "[경고] 거래월이 {}개다 ({}). 최근 3개월 전제가 깨져 제안을 만들 수 없다. "
            "dummy_data.py의 date를 확인하라."
        ).format(len(months), ", ".join(sorted(months)))

    this_month = max(months)
    monthly_totals = sum_by_month_category(labeled)
    savings = calc_savings(monthly_totals, labeled, this_month)

    return make_suggestion(savings)


def main():
    """
    Input  : 없음 (더미 데이터 사용)
    Process: run_pipeline으로 파이프라인을 한 번 실행한다
    Output : 제안 문장 1줄을 화면에 출력
    """
    print(run_pipeline(transactions, label_rules))

## 9. 실행 결과

아래 셀이 이 노트북의 최종 출력이다.


In [ ]:
main()

## 10. 테스트 케이스 결과

정상 / 후보 0개 / 값 누락 세 가지를 한 번에 출력해 결과표 작성에 쓴다. 담당: 고준호 · 안태현


In [ ]:
# 테스트 케이스 — 네 가지 입력을 같은 파이프라인에 넣어 결과를 비교한다
test_cases = [
    ("정상 (기본 더미 데이터)", transactions),
    ("엣지: 후보 0개", transactions_no_saving),
    ("엣지: 빈 입력", []),
    ("잘못된 입력: 값 누락", transactions_missing),
]

for name, data in test_cases:
    print(f"[{name}]")
    print(" ", run_pipeline(data, label_rules))
    print()
